In [1]:
#####  Import libraries for Boston City DoIT Analytics database..
import os 
import civis

# import libraries for data pre-processing
import pandas as pd
import numpy as np
from datetime import datetime
import datetime as datetime

# import libraries for visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from chart_studio import plotly
import plotly.offline as pyoff
import plotly.graph_objs as go
import plotly.express as px

# import libraries for geospatial analysis
import geopandas as gpd
import splot

In [2]:
os.environ['PYTHONPATH']='.'
os.environ['CIVIS_API_KEY']='BMHcrOhepkU6Wa-nZocM4Baf8IxVDecotXGyz8FLx14'

# Define the SQL query for proactive inspections from 2020 - present 
sql_proactive_inspections = """
SELECT * FROM isd_internal_data.proactive_inspections 
WHERE resultdttm >= '2020-01-01 00:00:00+00'
"""
# Define the SQL query for proactive violations from 2020 - present 
sql_proactive_violations = """
SELECT * FROM isd_internal_data.proactive_violations
WHERE resultdttm >= '2020-01-01 00:00:00+00'
"""

# Define the SQL query for reactive inspections/cases from 2023 - present 
sql_reactive_cases = """
SELECT * FROM isd_internal_data.reactive_cases
WHERE resultdttm >= '2023-01-01 00:00:00+00'
"""
# Define the SQL query for reactive violations from 2023 - present 
sql_reactive_violations = """
SELECT * FROM isd_internal_data.reactive_violations 
WHERE resdttm >= '2023-01-01 00:00:00+00'
"""

df_proactive_inspections = civis.io.read_civis_sql(sql_proactive_inspections,
                                            database='Boston',
                                            use_pandas=True)
df_proactive_violations = civis.io.read_civis_sql(sql_proactive_violations,
                                            database='Boston',
                                            use_pandas=True)
df_reactive_inspections = civis.io.read_civis_sql(sql_reactive_cases,
                                            database='Boston',
                                            use_pandas=True)
df_reactive_violations = civis.io.read_civis_sql(sql_reactive_violations,
                                            database='Boston',
                                            use_pandas=True)


C:\Users\174006\boston-env\lib\site-packages\civis\io\_tables.py:348: DtypeWarning:

Columns (2,8,9,19,31,32,36,37,59,62,64) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\174006\boston-env\lib\site-packages\civis\io\_tables.py:348: DtypeWarning:

Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\174006\boston-env\lib\site-packages\civis\io\_tables.py:348: DtypeWarning:

Columns (0,2,24,30,34,38,74,76,78,87,88,89,90) have mixed types. Specify dtype option on import or set low_memory=False.



In [3]:
df_proactive_inspections.head()


,licensecat,licenseno,comments_licappl,expdttm,issby,issdttm,modby_licappl,moddttm_licappl,licenseby,addby_licappl,...,gpsx,gpsy,gpsz,postdir,state,stname,stno,suffix,zip,gps_geom_2249
0,Condo,RINS-512542,NaN,2027-09-13 04:00:00+00,,NaN,HSNINTFC,2022-11-10 15:44:30.85+00,110163,110163,...,NaN,NaN,NaN,,,,,,,NaN
1,Condo,RINS-512543,NaN,2027-09-13 04:00:00+00,,NaN,HSNINTFC,2022-11-10 15:44:33.367+00,110163,110163,...,NaN,NaN,NaN,,,,,,,NaN
2,MultiFam,RINS-512544,NaN,2027-09-13 04:00:00+00,,NaN,HSNINTFC,2022-11-10 15:44:21.67+00,110163,110163,...,NaN,NaN,NaN,,,,,,,NaN
3,Condo,RINS-513784,"32 Sidlaw Road, Unit 11",2027-08-19 04:00:00+00,,NaN,HSNINTFC,2022-11-16 12:42:37.23+00,110163,110163,...,NaN,NaN,NaN,,,,,,,NaN
4,MultiFam,RINS-479165,NaN,2027-02-14 05:00:00+00,,NaN,HSNINTFC,2022-05-02 11:51:02.667+00,110163,110163,...,NaN,NaN,NaN,,,,,,,NaN


In [280]:
sub = df_proactive_inspections[['apkey','licensecat','licenseno','addrkey','result','resultdttm','resultby','compdttm','descript','gpsx','gpsy','gps_geom_2249']]


In [281]:
# Convert the pandas DataFrame to a geopandas GeoDataFrame
gdf = gpd.GeoDataFrame(sub, geometry=gpd.points_from_xy(sub.gpsx,sub.gpsy, crs=4326))

gdf.head()


,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry
0,2631828,Condo,RINS-512542,1,Pass,2022-09-13 04:00:00+00,110163,NaN,Rental Unit Inspection,NaN,NaN,NaN,POINT EMPTY
1,2631831,Condo,RINS-512543,1,Pass,2022-09-13 04:00:00+00,110163,NaN,Rental Unit Inspection,NaN,NaN,NaN,POINT EMPTY
2,2631833,MultiFam,RINS-512544,1,Pass,2022-09-13 04:00:00+00,110163,NaN,Rental Unit Inspection,NaN,NaN,NaN,POINT EMPTY
3,2634600,Condo,RINS-513784,1,Pass,2022-08-19 04:00:00+00,110163,NaN,Rental Unit Inspection,NaN,NaN,NaN,POINT EMPTY
4,2485376,MultiFam,RINS-479165,1,Pass,2022-02-14 05:00:00+00,110163,NaN,Rental Unit Inspection,NaN,NaN,NaN,POINT EMPTY


In [282]:
# Define a function to check if the geometry is empty or contains 'POINT EMPTY'.
def is_invalid_geometry(geom):
    return geom.is_empty or geom.geom_type == 'POINT EMPTY'

# Use the defined function to create a boolean mask for invalid geometries and NaN values.
mask_invalid_geometries = gdf['geometry'].apply(is_invalid_geometry)
mask_nan_values = gdf['geometry'].isna()

# Combine the boolean masks using logical OR (|).
mask_invalid_rows = mask_invalid_geometries | mask_nan_values

# Use the mask to filter out the rows with invalid geometries and NaN values.
gdf = gdf[~mask_invalid_rows]
gdf.head(10)


,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry
7,1825020,MultiFam,RINS-382145,639200,Pass,2020-03-06 20:42:00+00,130902,2020-03-06 20:42:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
8,1825019,MultiFam,RINS-382144,639200,Pass,2020-10-15 17:09:00+00,050242,2020-10-15 17:09:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
9,1825019,MultiFam,RINS-382144,639200,Pass,2020-03-06 16:37:00+00,130902,2020-03-06 16:37:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
10,1825019,MultiFam,RINS-382144,639200,Fail,2020-03-06 20:43:00+00,130902,2020-03-06 20:43:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
11,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
12,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
13,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
14,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
15,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)
16,1825016,MultiFam,RINS-382141,639200,Fail,2020-03-06 20:41:00+00,130902,2020-03-06 20:41:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758)


In [283]:
import pyproj

# Define the EPSG code for the UTM coordinate system used in the 'gpsx' and 'gpsy' columns.
epsg_code_utm = '32618'

# Define the EPSG code for the WGS 84 (latitude-longitude) coordinate system.
epsg_code_wgs84 = '4326'

# Create a Proj transformer to convert from UTM to WGS 84.
transformer = pyproj.Transformer.from_crs(epsg_code_utm, epsg_code_wgs84, always_xy=True)

# Convert 'gpsx' and 'gpsy' to latitude (X) and longitude (Y) columns.
gdf['X'], gdf['Y'] = transformer.transform(gdf['gpsx'].values, gdf['gpsy'].values)
gdf.head()


,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry,X,Y
7,1825020,MultiFam,RINS-382145,639200,Pass,2020-03-06 20:42:00+00,130902,2020-03-06 20:42:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
8,1825019,MultiFam,RINS-382144,639200,Pass,2020-10-15 17:09:00+00,050242,2020-10-15 17:09:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
9,1825019,MultiFam,RINS-382144,639200,Pass,2020-03-06 16:37:00+00,130902,2020-03-06 16:37:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
10,1825019,MultiFam,RINS-382144,639200,Fail,2020-03-06 20:43:00+00,130902,2020-03-06 20:43:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
11,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508


In [284]:
gdf.reset_index(drop=True, inplace=True)
gdf.reset_index(inplace=True)
gdf.rename(columns={'index':'ID'}, inplace=True)
gdf.head()


,ID,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry,X,Y
0,0,1825020,MultiFam,RINS-382145,639200,Pass,2020-03-06 20:42:00+00,130902,2020-03-06 20:42:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
1,1,1825019,MultiFam,RINS-382144,639200,Pass,2020-10-15 17:09:00+00,050242,2020-10-15 17:09:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
2,2,1825019,MultiFam,RINS-382144,639200,Pass,2020-03-06 16:37:00+00,130902,2020-03-06 16:37:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
3,3,1825019,MultiFam,RINS-382144,639200,Fail,2020-03-06 20:43:00+00,130902,2020-03-06 20:43:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508
4,4,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508


* Assume we wanna check whether there is a relationship between the number of units/apts in close proximity and the violations at each units/apts. 
* We begin analyzing this through the use of spatial proximity features.


In [285]:
import geopandas as gpd

shapefile_path = r"C:\Users\174006\Desktop\Boston\Boston_Neighborhoods.shp"
boston = gpd.read_file(shapefile_path)

# If the CRS is in geographic (latitude-longitude) coordinates, we can find an appropriate projected CRS for Boston.
if boston.crs.is_geographic:
    # Define the UTM zone for Boston. Boston is mostly within UTM Zone 19N.
    # You can choose a different UTM zone if required based on the exact location of your data.
    utm_zone = 19

    # Set the EPSG code for the projected CRS based on the UTM zone.
    target_crs = f'EPSG:326{utm_zone:02d}'

    # Reproject the GeoDataFrame to the target projected CRS.
    boston_projected = boston.to_crs(target_crs)

    # Now, 'boston_projected' contains the GeoDataFrame with geometries in the projected CRS.
    # You can use 'boston_projected' for further analysis or plotting.
    print(boston_projected.crs)

# Output will show the new projected CRS of the 'boston_projected' GeoDataFrame.


EPSG:32619


In [286]:
# Reproject 'gdf' to EPSG:32619 (UTM Zone 19N) to create a buffer in meters
# Define the projected CRS for Boston (EPSG:32619).
target_crs = 'EPSG:32619'

# Reproject the 'gdf' GeoDataFrame to the target CRS (EPSG:32619).
gdf_projected = gdf.to_crs(target_crs)


In [288]:
# Define the buffer distance in the units of the projected CRS.
# For example, if 'boston_projected' is in meters, and you want a buffer of 500 meters, set the distance as follows:
buffer_distance_meters = 100

# Creating a copy of the original dataframe to operate on
gdf_buffer = gdf.copy()

# Create a buffer around the geometries in 'boston_projected'.
gdf_buffer["buffer_5m"] = gdf_projected.buffer(buffer_distance_meters)
gdf_buffer.head()


,ID,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry,X,Y,buffer_5m
0,0,1825020,MultiFam,RINS-382145,639200,Pass,2020-03-06 20:42:00+00,130902,2020-03-06 20:42:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,POLYGON EMPTY
1,1,1825019,MultiFam,RINS-382144,639200,Pass,2020-10-15 17:09:00+00,050242,2020-10-15 17:09:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,POLYGON EMPTY
2,2,1825019,MultiFam,RINS-382144,639200,Pass,2020-03-06 16:37:00+00,130902,2020-03-06 16:37:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,POLYGON EMPTY
3,3,1825019,MultiFam,RINS-382144,639200,Fail,2020-03-06 20:43:00+00,130902,2020-03-06 20:43:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,POLYGON EMPTY
4,4,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,POLYGON EMPTY


In [279]:
# Define the buffer distance in the units of the projected CRS.
# For example, if 'boston_projected' is in meters, and you want a buffer of 500 meters, set the distance as follows:
buffer_distance_meters = 500

# Create a buffer around the geometries in 'boston_projected'.
gdf_buffer = gdf_projected.buffer(buffer_distance_meters)

# Now, 'boston_buffered' is a new GeoSeries with the buffered geometries.
# You can convert it back to a GeoDataFrame if needed:
buffer_gdf = gpd.GeoDataFrame(geometry=gdf_buffer, crs="EPSG:32619")
buffer_gdf.head()


,geometry
0,POLYGON EMPTY
1,POLYGON EMPTY
2,POLYGON EMPTY
3,POLYGON EMPTY
4,POLYGON EMPTY


In [289]:
# For example, if 'boston_projected' is in meters, and you want a buffer of 500 meters, set the distance as follows:
buffer_distance_meters = 500

# Create a buffer around the geometries in 'boston_projected'.
boston_buffered = boston_projected.buffer(buffer_distance_meters)

# Now, 'boston_buffered' is a new GeoSeries with the buffered geometries.
# You can convert it back to a GeoDataFrame if needed:
boston_buffered_gdf = gpd.GeoDataFrame(geometry=boston_buffered, crs=boston_projected.crs)
boston_buffered_gdf.head()


In [230]:
# Rename the 'geometry' column in 'boston_buffered_gdf' to 'buffered'.
boston_buffered_gdf.rename(columns={'geometry': 'buffered'}, inplace=True)

# Concatenate 'gdf' with 'boston_buffered_gdf' to include the 'buffered' column.
gdf = gpd.GeoDataFrame(pd.concat([gdf, boston_buffered_gdf['buffered']], axis=1), crs=gdf.crs)
gdf.head()


,ID,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry,X,Y,buffered
0,0,1825020,MultiFam,RINS-382145,639200,Pass,2020-03-06 20:42:00+00,130902,2020-03-06 20:42:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,"POLYGON ((322552.683 4684200.685, 322561.963 4..."
1,1,1825019,MultiFam,RINS-382144,639200,Pass,2020-10-15 17:09:00+00,050242,2020-10-15 17:09:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,"POLYGON ((323162.076 4685787.995, 323163.712 4..."
2,2,1825019,MultiFam,RINS-382144,639200,Pass,2020-03-06 16:37:00+00,130902,2020-03-06 16:37:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,"POLYGON ((325392.564 4688858.682, 325392.614 4..."
3,3,1825019,MultiFam,RINS-382144,639200,Fail,2020-03-06 20:43:00+00,130902,2020-03-06 20:43:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,"POLYGON ((325595.340 4689609.217, 325597.986 4..."
4,4,1825016,MultiFam,RINS-382141,639200,Pass,2021-04-08 19:52:00+00,051104,2021-04-08 19:52:00+00,Rental Unit Inspection,770040.050503,2.948933e+06,0101000020C90800004D7BDB19F07F2741B8A38743A27F...,POINT (770040.05050 2948932.52758),-72.287542,26.635508,"POLYGON ((328992.286 4690911.239, 328996.981 4..."


In [135]:
joined.head()

,ID,apkey,licensecat,licenseno,addrkey,result,resultdttm,resultby,compdttm,descript,gpsx,gpsy,gps_geom_2249,geometry,X,Y,buffered,buffered,index_right


In [ ]:
# proactive count
proactive_count = (joined.groupby("ID_left").count())

# Converting to a dataframe and cleaning up
proactive_count_df = proactive_count.reset_index()
proactive_count_df = proactive_count_df[['ID_left','ID_right']]
proactive_count_df.columns=['ID','Location_Count']

proactive_count_df.head()


In [93]:
# Define the buffer distance in the units of the projected CRS (EPSG:32619 uses meters).
buffer_distance_meters = 500

# Reproject 'gdf' to EPSG:32619 (UTM Zone 19N) to create a buffer in meters.
gdf = gdf.to_crs('EPSG:32619')

# Create a buffer around the geometries in 'gdf'.
gdf['buffer_boston'] = gdf.geometry.buffer(buffer_distance_meters)

# Display the output with the specified columns ('ID', 'X', 'Y', 'geometry', 'buffer_boston').
print(gdf[['ID', 'X', 'Y', 'geometry', 'buffer_boston']])


            ID          X          Y         geometry  buffer_boston
0            7 -72.287542  26.635508  POINT (inf inf)  POLYGON EMPTY
1            8 -72.287542  26.635508  POINT (inf inf)  POLYGON EMPTY
2            9 -72.287542  26.635508  POINT (inf inf)  POLYGON EMPTY
3           10 -72.287542  26.635508  POINT (inf inf)  POLYGON EMPTY
4           11 -72.287542  26.635508  POINT (inf inf)  POLYGON EMPTY
...        ...        ...        ...              ...            ...
146200  146567 -72.230368  26.696706  POINT (inf inf)  POLYGON EMPTY
146201  146568 -72.230368  26.696706  POINT (inf inf)  POLYGON EMPTY
146202  146569 -72.230368  26.696706  POINT (inf inf)  POLYGON EMPTY
146203  146570 -72.230368  26.696706  POINT (inf inf)  POLYGON EMPTY
146204  146571 -72.230368  26.696706  POINT (inf inf)  POLYGON EMPTY

[146205 rows x 5 columns]


Buffered locations

The next step is to join the newly created buffered data to the original file. This will combine the two datasets based on geometry and allow us to perform an aggregation function:


In [ ]:
# Joining the buffer to the store locations table

# Right table is the proactive data
# Left table is that of the buffers around the locations
# The operation, or spatial predicate, you'll use is "within"

joined = gpd.sjoin(gdf,boston_buffered_gdf.set_geometry("geometry")[["ID", "geometry"]],predicate="within")
